In [1]:
!pip install pandas numpy pydantic requests -q


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import json
import hashlib
import re

from pathlib import Path
from pydantic import BaseModel, Field, ConfigDict

In [3]:
base_path = Path("metricmind_week3")

audit_path = base_path / "audit"
api_path = base_path / "api"
output_path = base_path / "outputs"

audit_path.mkdir(parents=True, exist_ok=True)
api_path.mkdir(parents=True, exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

print("Week 3 folders created successfully.")
print("Base folder:", base_path)
print("Audit folder:", audit_path)
print("API folder:", api_path)
print("Output folder:", output_path)

Week 3 folders created successfully.
Base folder: metricmind_week3
Audit folder: metricmind_week3\audit
API folder: metricmind_week3\api
Output folder: metricmind_week3\outputs


In [4]:
sales_data = pd.DataFrame({
    "date": [
        "2026-07-01",
        "2026-07-15",
        "2026-08-01",
        "2026-08-15",
        "2026-09-01",
        "2026-09-15",
        "2026-09-30",
        "2026-10-01",
        "2026-10-15",
        "2026-11-01"
    ],

    "region": [
        "Europe",
        "Europe",
        "Europe",
        "Asia",
        "Europe",
        "Europe",
        "North America",
        "Asia",
        "Europe",
        "Europe"
    ],

    "country": [
        "Germany",
        "France",
        "Germany",
        "India",
        "France",
        "Spain",
        "USA",
        "India",
        "Italy",
        "Germany"
    ],

    "revenue": [
        12000,
        15000,
        18000,
        10000,
        22000,
        17000,
        25000,
        14000,
        16000,
        19000
    ],

    "cost": [
        7000,
        8500,
        10000,
        6000,
        12000,
        9500,
        14000,
        8000,
        9000,
        10500
    ]
})

sales_data["date"] = pd.to_datetime(sales_data["date"])

print("Deterministic sales dataset created.")
display(sales_data)

Deterministic sales dataset created.


,date,region,country,revenue,cost
0,2026-07-01,Europe,Germany,12000,7000
1,2026-07-15,Europe,France,15000,8500
2,2026-08-01,Europe,Germany,18000,10000
3,2026-08-15,Asia,India,10000,6000
4,2026-09-01,Europe,France,22000,12000
5,2026-09-15,Europe,Spain,17000,9500
6,2026-09-30,North America,USA,25000,14000
7,2026-10-01,Asia,India,14000,8000
8,2026-10-15,Europe,Italy,16000,9000
9,2026-11-01,Europe,Germany,19000,10500


In [6]:
# Make sure date is in datetime format
sales_data["date"] = pd.to_datetime(sales_data["date"])

# Create quarter column
sales_data["quarter"] = sales_data["date"].dt.to_period("Q").astype(str)

print("Quarter column created successfully.")

display(
    sales_data[
        ["date", "region", "country", "revenue", "cost", "quarter"]
    ]
)

Quarter column created successfully.


,date,region,country,revenue,cost,quarter
0,2026-07-01,Europe,Germany,12000,7000,2026Q3
1,2026-07-15,Europe,France,15000,8500,2026Q3
2,2026-08-01,Europe,Germany,18000,10000,2026Q3
3,2026-08-15,Asia,India,10000,6000,2026Q3
4,2026-09-01,Europe,France,22000,12000,2026Q3
5,2026-09-15,Europe,Spain,17000,9500,2026Q3
6,2026-09-30,North America,USA,25000,14000,2026Q3
7,2026-10-01,Asia,India,14000,8000,2026Q4
8,2026-10-15,Europe,Italy,16000,9000,2026Q4
9,2026-11-01,Europe,Germany,19000,10500,2026Q4


In [7]:
q3_data = sales_data[
    sales_data["quarter"] == "2026Q3"
]

print("Q3 2026 records:")
display(q3_data)

print("\nNumber of Q3 records:", len(q3_data))

Q3 2026 records:


,date,region,country,revenue,cost,quarter
0,2026-07-01,Europe,Germany,12000,7000,2026Q3
1,2026-07-15,Europe,France,15000,8500,2026Q3
2,2026-08-01,Europe,Germany,18000,10000,2026Q3
3,2026-08-15,Asia,India,10000,6000,2026Q3
4,2026-09-01,Europe,France,22000,12000,2026Q3
5,2026-09-15,Europe,Spain,17000,9500,2026Q3
6,2026-09-30,North America,USA,25000,14000,2026Q3



Number of Q3 records: 7


In [8]:
semantic_definition = {
    "cube": "sales",

    "measures": {
        "revenue": {
            "sql": "SUM(revenue)",
            "type": "sum"
        },
        "cost": {
            "sql": "SUM(cost)",
            "type": "sum"
        }
    },

    "dimensions": {
        "date": {
            "type": "time"
        },
        "region": {
            "type": "string"
        },
        "country": {
            "type": "string"
        }
    }
}

print(json.dumps(semantic_definition, indent=2))

{
  "cube": "sales",
  "measures": {
    "revenue": {
      "sql": "SUM(revenue)",
      "type": "sum"
    },
    "cost": {
      "sql": "SUM(cost)",
      "type": "sum"
    }
  },
  "dimensions": {
    "date": {
      "type": "time"
    },
    "region": {
      "type": "string"
    },
    "country": {
      "type": "string"
    }
  }
}


In [9]:
def calculate_q3_revenue(data):
    q3 = data[
        data["quarter"] == "2026Q3"
    ]

    revenue = q3["revenue"].sum()

    return int(revenue)


official_q3_revenue = calculate_q3_revenue(sales_data)

print("Official Q3 Revenue:", official_q3_revenue)

Official Q3 Revenue: 119000


In [10]:
q3_revenue_values = q3_data["revenue"].tolist()

print("Q3 Revenue values:")
print(q3_revenue_values)

print("\nCalculation:")
print(" + ".join(map(str, q3_revenue_values)))

print("\nQ3 Revenue =", official_q3_revenue)

Q3 Revenue values:
[12000, 15000, 18000, 10000, 22000, 17000, 25000]

Calculation:
12000 + 15000 + 18000 + 10000 + 22000 + 17000 + 25000

Q3 Revenue = 119000


In [11]:
approved_q3_query = {
    "measures": [
        "sales.revenue"
    ],

    "dimensions": [],

    "timeDimensions": [
        {
            "dimension": "sales.date",
            "dateRange": [
                "2026-07-01",
                "2026-09-30"
            ]
        }
    ],

    "filters": []
}

print(json.dumps(
    approved_q3_query,
    indent=2
))

{
  "measures": [
    "sales.revenue"
  ],
  "dimensions": [],
  "timeDimensions": [
    {
      "dimension": "sales.date",
      "dateRange": [
        "2026-07-01",
        "2026-09-30"
      ]
    }
  ],
  "filters": []
}


In [12]:
def query_fingerprint(query):
    canonical_query = json.dumps(
        query,
        sort_keys=True,
        separators=(",", ":")
    )

    return hashlib.sha256(
        canonical_query.encode("utf-8")
    ).hexdigest()


q3_query_hash = query_fingerprint(
    approved_q3_query
)

print("Q3 Query Fingerprint:")
print(q3_query_hash)

Q3 Query Fingerprint:
95fb8b7b3e4c0469fe6871a81b0d2095f8d1ef83f3bb4a4556d734daef2d90a7


In [13]:
number_of_runs = 10

audit_results = []

for run_number in range(1, number_of_runs + 1):

    result = calculate_q3_revenue(sales_data)

    audit_results.append({
        "run": run_number,
        "query": "Q3 Revenue",
        "result": result
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

,run,query,result
0,1,Q3 Revenue,119000
1,2,Q3 Revenue,119000
2,3,Q3 Revenue,119000
3,4,Q3 Revenue,119000
4,5,Q3 Revenue,119000
5,6,Q3 Revenue,119000
6,7,Q3 Revenue,119000
7,8,Q3 Revenue,119000
8,9,Q3 Revenue,119000
9,10,Q3 Revenue,119000


In [14]:
unique_results = audit_df["result"].unique()

print("Unique results:", unique_results)

if len(unique_results) == 1:
    print("\nPASS: Every Q3 Revenue request returned the same value.")
else:
    print("\nFAIL: Results are inconsistent.")

Unique results: [119000]

PASS: Every Q3 Revenue request returned the same value.


In [15]:
expected_value = official_q3_revenue

matching_runs = (
    audit_df["result"] == expected_value
).sum()

repeatability_percentage = (
    matching_runs / number_of_runs
) * 100

print("Expected Q3 Revenue:", expected_value)
print("Matching runs:", matching_runs)
print("Total runs:", number_of_runs)
print("Repeatability:", repeatability_percentage, "%")

Expected Q3 Revenue: 119000
Matching runs: 10
Total runs: 10
Repeatability: 100.0 %


In [16]:
governance_passed = (
    audit_df["result"].nunique() == 1
    and audit_df["result"].iloc[0] == official_q3_revenue
)

if governance_passed:
    governance_status = "PASS"
else:
    governance_status = "FAIL"

governance_report = {
    "test": "Q3 Revenue Repeatability",
    "runs": number_of_runs,
    "expected_result": official_q3_revenue,
    "unique_results": int(audit_df["result"].nunique()),
    "repeatability_percentage": repeatability_percentage,
    "status": governance_status,
    "query_fingerprint": q3_query_hash
}

print(json.dumps(
    governance_report,
    indent=2
))

{
  "test": "Q3 Revenue Repeatability",
  "runs": 10,
  "expected_result": 119000,
  "unique_results": 1,
  "repeatability_percentage": 100.0,
  "status": "PASS",
  "query_fingerprint": "95fb8b7b3e4c0469fe6871a81b0d2095f8d1ef83f3bb4a4556d734daef2d90a7"
}


In [17]:
allowed_measures = {
    "sales.revenue",
    "sales.cost"
}

allowed_dimensions = {
    "sales.date",
    "sales.region",
    "sales.country"
}

print("Allowed measures:")
print(allowed_measures)

print("\nAllowed dimensions:")
print(allowed_dimensions)

Allowed measures:
{'sales.cost', 'sales.revenue'}

Allowed dimensions:
{'sales.country', 'sales.date', 'sales.region'}


In [18]:
def validate_query(query):

    errors = []

    for measure in query.get("measures", []):

        if measure not in allowed_measures:
            errors.append(
                f"Unauthorized measure: {measure}"
            )

    for dimension in query.get("dimensions", []):

        if dimension not in allowed_dimensions:
            errors.append(
                f"Unauthorized dimension: {dimension}"
            )

    for time_dimension in query.get(
        "timeDimensions", []
    ):

        dimension = time_dimension["dimension"]

        if dimension not in allowed_dimensions:
            errors.append(
                f"Unauthorized time dimension: {dimension}"
            )

    return errors


validation_result = validate_query(
    approved_q3_query
)

if not validation_result:
    print("PASS: Query uses only approved semantic-layer fields.")
else:
    print("FAIL:")
    for error in validation_result:
        print("-", error)

PASS: Query uses only approved semantic-layer fields.


In [19]:
hallucinated_query = {
    "measures": [
        "sales.profit"
    ],

    "dimensions": [
        "sales.customer_lifetime_value"
    ],

    "timeDimensions": [],

    "filters": []
}

hallucination_errors = validate_query(
    hallucinated_query
)

print("Testing intentionally invalid query...\n")

if hallucination_errors:

    print("PASS: Hallucinated fields were blocked.")

    for error in hallucination_errors:
        print("-", error)

else:
    print("FAIL: Hallucinated query was accepted.")

Testing intentionally invalid query...

PASS: Hallucinated fields were blocked.
- Unauthorized measure: sales.profit
- Unauthorized dimension: sales.customer_lifetime_value


In [20]:
european_sales_payload = {
    "measures": [
        "sales.revenue"
    ],

    "dimensions": [
        "sales.country"
    ],

    "timeDimensions": [],

    "filters": [
        {
            "member": "sales.region",
            "operator": "equals",
            "values": [
                "Europe"
            ]
        }
    ]
}

print(json.dumps(
    european_sales_payload,
    indent=2
))

{
  "measures": [
    "sales.revenue"
  ],
  "dimensions": [
    "sales.country"
  ],
  "timeDimensions": [],
  "filters": [
    {
      "member": "sales.region",
      "operator": "equals",
      "values": [
        "Europe"
      ]
    }
  ]
}


In [21]:
api_validation_errors = validate_query(
    european_sales_payload
)

if api_validation_errors:

    print("FAIL: API payload contains invalid fields.")

    for error in api_validation_errors:
        print("-", error)

else:

    print("PASS: European sales payload is valid.")
    print("\nPayload:")
    print(
        json.dumps(
            european_sales_payload,
            indent=2
        )
    )

PASS: European sales payload is valid.

Payload:
{
  "measures": [
    "sales.revenue"
  ],
  "dimensions": [
    "sales.country"
  ],
  "timeDimensions": [],
  "filters": [
    {
      "member": "sales.region",
      "operator": "equals",
      "values": [
        "Europe"
      ]
    }
  ]
}


In [22]:
def semantic_layer_api(payload):

    errors = validate_query(payload)

    if errors:

        return {
            "status": "error",
            "errors": errors
        }

    result = sales_data.copy()

    filters = payload.get("filters", [])

    for filter_item in filters:

        member = filter_item["member"]
        operator = filter_item["operator"]
        values = filter_item["values"]

        if member == "sales.region" and operator == "equals":

            result = result[
                result["region"].isin(values)
            ]

    response = {}

    if "sales.revenue" in payload["measures"]:

        response["revenue"] = int(
            result["revenue"].sum()
        )

    if "sales.cost" in payload["measures"]:

        response["cost"] = int(
            result["cost"].sum()
        )

    if "sales.country" in payload["dimensions"]:

        response["data"] = (
            result.groupby("country")["revenue"]
            .sum()
            .reset_index()
            .to_dict(orient="records")
        )

    return {
        "status": "success",
        "data": response
    }


api_response = semantic_layer_api(
    european_sales_payload
)

print(json.dumps(
    api_response,
    indent=2
))

{
  "status": "success",
  "data": {
    "revenue": 119000,
    "data": [
      {
        "country": "France",
        "revenue": 37000
      },
      {
        "country": "Germany",
        "revenue": 49000
      },
      {
        "country": "Italy",
        "revenue": 16000
      },
      {
        "country": "Spain",
        "revenue": 17000
      }
    ]
  }
}


In [23]:
print("=" * 70)
print("METRICMIND WEEK 3 FINAL TEST")
print("=" * 70)

print("\n1. GOVERNANCE AUDIT")

if governance_passed:
    print("PASS")
    print(
        f"Q3 Revenue = {official_q3_revenue}"
    )
    print(
        f"Repeatability = {repeatability_percentage}%"
    )
else:
    print("FAIL")


print("\n2. SQL HALLUCINATION PROTECTION")

if hallucination_errors:
    print("PASS")
else:
    print("FAIL")


print("\n3. EUROPEAN SALES API CHECK")

if not api_validation_errors:
    print("PASS")
else:
    print("FAIL")


print("\n4. API RESPONSE")

print(
    json.dumps(
        api_response,
        indent=2
    )
)

METRICMIND WEEK 3 FINAL TEST

1. GOVERNANCE AUDIT
PASS
Q3 Revenue = 119000
Repeatability = 100.0%

2. SQL HALLUCINATION PROTECTION
PASS

3. EUROPEAN SALES API CHECK
PASS

4. API RESPONSE
{
  "status": "success",
  "data": {
    "revenue": 119000,
    "data": [
      {
        "country": "France",
        "revenue": 37000
      },
      {
        "country": "Germany",
        "revenue": 49000
      },
      {
        "country": "Italy",
        "revenue": 16000
      },
      {
        "country": "Spain",
        "revenue": 17000
      }
    ]
  }
}


In [24]:
week3_report = {
    "project": "MetricMind",
    "week": 3,

    "governance_audit": {
        "test": "Q3 Revenue Repeatability",
        "runs": number_of_runs,
        "expected_result": official_q3_revenue,
        "unique_results": int(
            audit_df["result"].nunique()
        ),
        "repeatability_percentage":
            repeatability_percentage,
        "status": governance_status,
        "query_fingerprint": q3_query_hash
    },

    "sql_hallucination_protection": {
        "status": (
            "PASS"
            if hallucination_errors
            else "FAIL"
        ),
        "blocked_fields": [
            "sales.profit",
            "sales.customer_lifetime_value"
        ]
    },

    "api_check": {
        "question": "Show me European sales",
        "status": (
            "PASS"
            if not api_validation_errors
            else "FAIL"
        ),
        "payload": european_sales_payload,
        "response": api_response
    }
}

report_file = output_path / "week3_governance_report.json"

with open(
    report_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        week3_report,
        file,
        indent=2
    )

print("=" * 70)
print("METRICMIND WEEK 3 COMPLETED")
print("=" * 70)

print("\nGovernance Audit:", governance_status)

print(
    "Q3 Revenue:",
    official_q3_revenue
)

print(
    "Repeatability:",
    repeatability_percentage,
    "%"
)

print(
    "SQL Hallucination Protection:",
    "PASS" if hallucination_errors else "FAIL"
)

print(
    "European Sales API Check:",
    "PASS" if not api_validation_errors else "FAIL"
)

print("\nReport saved to:")
print(report_file)

METRICMIND WEEK 3 COMPLETED

Governance Audit: PASS
Q3 Revenue: 119000
Repeatability: 100.0 %
SQL Hallucination Protection: PASS
European Sales API Check: PASS

Report saved to:
metricmind_week3\outputs\week3_governance_report.json
